# EvoMind — Phase 5: QLoRA fine-tune on the agent's accumulated mind

Run this notebook on **Google Colab** (T4) or **Kaggle** (T4 ×2 / P100). Both free tiers can fine-tune a 7-8B parameter model with QLoRA in roughly 45–90 min for ~2,000 examples.

## What this does

1. Pulls the training corpus from your local EvoMind API (`/api/export/training-corpus`) — or you can upload the JSON if your API isn't internet-reachable.
2. Loads **Llama-3-8B-Instruct** in 4-bit (bitsandbytes) — fits in 16 GB VRAM with margin.
3. Trains a LoRA adapter on the (instruction, input, output) triples for ~3 epochs.
4. Saves the adapter to Drive (Colab) or the Kaggle output directory.
5. Optionally pushes the adapter to a private Hugging Face repo.

## What this does NOT do

- It does not give the model memory. Memory lives in your local SQLite database, retrieved at inference time. Fine-tuning bakes in *style* and *common-domain reasoning patterns*, not facts.
- It does not host an inference endpoint. After training, download the adapter and run it locally with `transformers` + `peft`, or merge into a GGUF for Ollama / llama.cpp.

## Threshold reminder

Below 1,000 high-quality (question, answer) pairs, fine-tuning is mostly noise. The export endpoint tells you whether you're ready.

## Step 1 — Install (Colab/Kaggle, ~2 min)

In [ ]:
!pip install -q -U transformers==4.45.2 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1 datasets==3.0.1 trl==0.11.4

## Step 2 — Get the training corpus

Two paths:

**A. If your local EvoMind is reachable from Colab (e.g. via ngrok or a public IP):**
```python
import requests
API = 'https://your-evomind.ngrok.app'  # or the public URL
data = requests.get(f'{API}/api/export/training-corpus?format=alpaca&min_confidence=0.65').json()
print(data['recommendation'])
rows = data['rows']
```

**B. The reliable path — manually export and upload:**
On your local machine run:
```
curl 'http://127.0.0.1:8000/api/export/training-corpus?format=alpaca&min_confidence=0.65' > corpus.json
```
Then upload `corpus.json` to this Colab session via the file panel, or to your Kaggle dataset.

In [ ]:
import json
from pathlib import Path

# Set this to wherever you uploaded corpus.json
CORPUS_PATH = Path('corpus.json')

with open(CORPUS_PATH) as f:
    payload = json.load(f)

rows = payload['rows']
print(f"Loaded {len(rows)} training examples")
print(f"Stats: {payload['stats']}")
print(f"Recommendation: {payload['recommendation']}")

if len(rows) < 200:
    raise ValueError('Corpus is too small. Drop more PDFs and let the autopilot run before fine-tuning.')

## Step 3 — Build the prompt template and tokenize

Using the official Llama-3 chat template so the fine-tuned model follows the same conventions the base model was trained with.

In [ ]:
from datasets import Dataset

SYSTEM = (
    'You are EvoMind, a research assistant grounded in evidence. '
    'Answer carefully, cite reasoning, and admit uncertainty. '
    'Format citations as [#index].'
)

def to_chat(ex):
    user_text = ex['instruction']
    if ex.get('input'):
        user_text += '\n\nEvidence:\n' + ex['input']
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content': user_text},
            {'role': 'assistant', 'content': ex['output']},
        ]
    }

ds = Dataset.from_list([to_chat(r) for r in rows])
ds = ds.train_test_split(test_size=0.05, seed=7)
print(ds)

## Step 4 — Load Llama-3-8B-Instruct in 4-bit

Other good choices:
- `mistralai/Mistral-7B-Instruct-v0.3` — a bit smaller, less restrictive license
- `microsoft/Phi-3-mini-4k-instruct` — 3.8B, very fast, 16 GB has lots of headroom
- `Qwen/Qwen2.5-7B-Instruct` — newer, often beats Llama-3 on reasoning

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'   # accept the gated license on HF first

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## Step 5 — Train (TRL SFTTrainer, ~30–60 min on T4)

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir='./evomind-lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=20,
    save_strategy='epoch',
    eval_strategy='epoch',
    max_seq_length=2048,
    packing=True,
    dataset_text_field=None,   # using messages field via apply_chat_template inside SFT
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    tokenizer=tokenizer,
)
trainer.train()

## Step 6 — Save adapter and quick smoke test

In [ ]:
trainer.save_model('./evomind-lora')
tokenizer.save_pretrained('./evomind-lora')
print('Adapter saved to ./evomind-lora — download this folder and use it locally.')

In [ ]:
# Quick smoke test: ask it a question that's adjacent to your corpus
test_prompt = 'What does the literature say about the relationship between attention and consciousness?'
inputs = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': test_prompt},
    ],
    return_tensors='pt', add_generation_prompt=True,
).to(model.device)

with torch.inference_mode():
    out = model.generate(inputs, max_new_tokens=400, do_sample=True, temperature=0.6)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## Step 7 — Use the adapter locally

Download the `evomind-lora` directory. To run inference on your local machine (CPU or modest GPU):

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct', torch_dtype='auto')
model = PeftModel.from_pretrained(base, './evomind-lora')
tok   = AutoTokenizer.from_pretrained('./evomind-lora')
```

Or merge + convert to GGUF for **Ollama**:
```bash
python -m peft.utils.merge_and_unload --adapter ./evomind-lora --base meta-llama/Meta-Llama-3-8B-Instruct --output ./evomind-merged
python convert_hf_to_gguf.py ./evomind-merged --outfile evomind.gguf --outtype q4_K_M
ollama create evomind -f Modelfile
```

Then point your local EvoMind at it by setting:
```
PRIMARY_PROVIDER=ollama
OLLAMA_MODEL=evomind
```